# Collatz: control de profundidad y escala ordinal (Kaggle, 2×T4)

Dos planes de `experimentos.py`, código embebido (sin internet):

- **profundidad** (9 corridas): una capa MPS, dos capas sin compuerta y dos capas con espigas, en T³.
  Responde si el 100 % de las espigas era de la profundidad o del silencio.
- **escala** (108 corridas): MPS complejo con χ = 8, 16, 32, 64 y Transformer d = 64, 128, en T¹..T⁶,
  3 semillas. Da el mayor k que cada tamaño resuelve exacto y general: la escala ordinal.

Ajustes: **Accelerator → GPU T4 x2**. Cada plan en su GPU, 2 procesos por GPU (los modelos son de
bucle en Python, la GPU no se satura). Estimación: profundidad ~30 min; escala 3 a 5 h (χ = 64 y k = 6
son las corridas lentas). Salida en `/kaggle/working/resultados/<plan>/*.json` y las tablas al final.
Si se corta, volver a ejecutar continúa donde iba: las corridas terminadas no se repiten.

In [ ]:
%%writefile collatz.py
"""Tareas de estrés sobre el mapa de Collatz, en bits LSB-primero.

Tareas:
  paso-k   : dado n en binario (L bits), emitir T^k(n) en binario (L + 2k bits). Etiquetado por posición.
             Para k fijo es una función de estado finito (transductor con acarreo), así que un modelo
             con memoria acotada puede representarla exactamente para cualquier L.
  parada   : dado n, decidir si su tiempo total de parada supera la mediana de su longitud.
             No es de estado finito: es la frontera "Smale 18" del banco de pruebas.
"""
from __future__ import annotations

import numpy as np

PAD = 2  # símbolo de relleno en la entrada (vocabulario: 0, 1, PAD)


def T(n: int) -> int:
    return n // 2 if n % 2 == 0 else 3 * n + 1


def T_k(n: int, k: int) -> int:
    for _ in range(k):
        n = T(n)
    return n


def tiempo_parada(n: int, tope: int = 10_000) -> int:
    s = 0
    while n != 1 and s < tope:
        n = T(n)
        s += 1
    return s


def a_bits(n: int, ancho: int) -> np.ndarray:
    """n → bits LSB-primero, ancho fijo."""
    return np.array([(n >> i) & 1 for i in range(ancho)], dtype=np.int64)


def muestras_n(rng: np.random.Generator, L: int, B: int) -> np.ndarray:
    """B enteros con exactamente L bits (bit L-1 encendido)."""
    if L < 63:
        return rng.integers(1 << (L - 1), 1 << L, size=B, dtype=np.int64)
    # más de 63 bits: enteros de Python (sin límite), bit alto encendido
    bajos = rng.integers(0, 2, size=(B, L - 1))
    return np.array([(1 << (L - 1)) | int("".join(map(str, fila)), 2) for fila in bajos], dtype=object)


def lote_paso_k(rng: np.random.Generator, L: int, B: int, k: int):
    """Entrada: n en L bits + relleno hasta L+2k. Salida: T^k(n) en L+2k bits."""
    ancho = L + 2 * k
    ns = muestras_n(rng, L, B)
    x = np.full((B, ancho), PAD, dtype=np.int64)
    y = np.zeros((B, ancho), dtype=np.int64)
    for i, n in enumerate(ns):
        x[i, :L] = a_bits(int(n), L)
        y[i] = a_bits(T_k(int(n), k), ancho)
    return x, y


_medianas: dict[int, float] = {}


def mediana_parada(L: int, rng_semilla: int = 12345, N: int = 4000) -> float:
    if L not in _medianas:
        rng = np.random.default_rng(rng_semilla + L)
        ns = muestras_n(rng, L, N)
        _medianas[L] = float(np.median([tiempo_parada(int(n)) for n in ns]))
    return _medianas[L]


def lote_parada(rng: np.random.Generator, L: int, B: int):
    """Entrada: n en L bits. Salida: 1 si tiempo de parada > mediana(L)."""
    ns = muestras_n(rng, L, B)
    med = mediana_parada(L)
    x = np.stack([a_bits(int(n), L) for n in ns])
    y = np.array([int(tiempo_parada(int(n)) > med) for n in ns], dtype=np.int64)
    return x, y


def lote(tarea: str, rng: np.random.Generator, L: int, B: int):
    if tarea.startswith("paso-"):
        return lote_paso_k(rng, L, B, int(tarea.split("-")[1]))
    if tarea == "parada":
        return lote_parada(rng, L, B)
    raise ValueError(tarea)


if __name__ == "__main__":
    rng = np.random.default_rng(0)
    x, y = lote_paso_k(rng, 6, 3, 1)
    for xi, yi in zip(x, y):
        n = sum(int(b) << i for i, b in enumerate(xi[:6]))
        m = sum(int(b) << i for i, b in enumerate(yi))
        assert m == T(n), (n, m)
        print(n, "→", m, xi, yi)
    x, y = lote_parada(rng, 10, 5)
    print("mediana(10) =", mediana_parada(10), "etiquetas", y)
    assert tiempo_parada(27) == 111
    print("ok")


In [ ]:
%%writefile modelos.py
"""Modelos del banco de pruebas de superposición.

La idea central, en su forma clásica y computable: el estado de la secuencia no es un vector de
activaciones sino una AMPLITUD sobre χ estados internos a la vez (superposición). Cada símbolo x
aplica una matriz A[x] ∈ C^{χ×χ}; la secuencia completa es un producto de matrices (MPS / red de
tensores) y la lectura sigue la regla de Born |amplitud|², de modo que las hipótesis internas
interfieren constructiva o destructivamente antes de emitir el bit.

Variantes (todas comparten el mismo esqueleto de contracción izquierda/derecha):
  mps-real      matrices reales, lectura lineal (sin interferencia).
  mps-complejo  matrices complejas libres, lectura de Born.
  mps-unitario  A[x] = Cayley(H[x]) unitarias: función de onda simulada de norma constante.
  mps-fourier   A[x] = V diag(e^{iθ[x]}) V†, base propia COMPARTIDA: todas las A conmutan.
                Es la "serie de Fourier" pura: el estado es Σ_k c_k e^{i Σ_t θ_k[x_t]}.
  Compuerta de espigas (LIF con gradiente sustituto) entre dos capas MPS.
  Dos cabezas con crítica cruzada: cada cabeza corrige su amplitud tras ver la de la otra,
                y la respuesta final es la interferencia de ambas.
Referencia: Transformer codificador (bidireccional) con RoPE o con ábaco (posiciones desplazadas
al azar en entrenamiento, como en navros/).
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB = 3  # 0, 1, PAD


# ----------------------------------------------------------------------------- utilidades complejas
def cayley(B: torch.Tensor) -> torch.Tensor:
    """B libre (compleja) → H = B + B† hermitiana → U = (I - iH)(I + iH)^{-1} unitaria."""
    H = B + B.conj().transpose(-1, -2)
    I = torch.eye(B.shape[-1], dtype=B.dtype, device=B.device)
    return torch.linalg.solve(I + 1j * H, I - 1j * H, left=False)


def _normaliza(v: torch.Tensor) -> torch.Tensor:
    return v / (v.norm(dim=-1, keepdim=True) + 1e-8)


# ----------------------------------------------------------------------------- capa MPS
class CapaMPS(nn.Module):
    """Contrae la secuencia por la izquierda y por la derecha y devuelve, por posición,
    las amplitudes (o valores reales) de cada clase: s_t[c] = L_t · M[c] · R_{t+1}.

    Si `entrada_continua`, la matriz de cada posición es A(f_t) = A_0 + Σ_j f_tj A_j
    (mezcla lineal de matrices) en vez de A[x_t]; sirve para apilar capas."""

    def __init__(self, chi: int, n_clases: int, variante: str, n_simbolos: int = VOCAB,
                 entrada_continua: bool = False, d_entrada: int = 0):
        super().__init__()
        self.chi, self.variante, self.n_clases = chi, variante, n_clases
        self.compleja = variante != "real"
        self.entrada_continua = entrada_continua
        dt = torch.cfloat if self.compleja else torch.float32
        n_mat = (1 + d_entrada) if entrada_continua else n_simbolos
        esc = 1.0 / math.sqrt(chi)
        self.A = nn.Parameter(torch.randn(n_mat, chi, chi, dtype=dt) * esc)
        if variante == "fourier":
            self.V = nn.Parameter(torch.randn(chi, chi, dtype=torch.cfloat) * esc)  # base propia compartida
            self.theta = nn.Parameter(torch.rand(n_mat, chi) * 2 * math.pi)        # fases por símbolo
        self.izq = nn.Parameter(torch.randn(chi, dtype=dt) * esc)
        self.der = nn.Parameter(torch.randn(chi, dtype=dt) * esc)
        self.M = nn.Parameter(torch.randn(n_clases, chi, chi, dtype=dt) * esc)

    def matrices(self) -> torch.Tensor:
        if self.variante in ("real", "complejo"):
            return self.A
        if self.variante == "unitario":
            return cayley(self.A)
        if self.variante == "fourier":
            V = cayley(self.V)
            D = torch.exp(1j * self.theta)  # (n_mat, chi)
            return V.unsqueeze(0) * D.unsqueeze(1) @ V.conj().transpose(-1, -2).unsqueeze(0)
        raise ValueError(self.variante)

    def por_posicion(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T) símbolos o (B, T, d) continuo → matrices (B, T, chi, chi)."""
        A = self.matrices()
        if not self.entrada_continua:
            return A[x]
        f = x.to(A.dtype) if self.compleja else x
        return A[0] + torch.einsum("ntj,jab->ntab", f, A[1:])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Devuelve s: (B, T, n_clases) amplitudes complejas (o reales en la variante real)."""
        Ax = self.por_posicion(x)  # (B, T, χ, χ)
        B, T = Ax.shape[:2]
        L = [_normaliza(self.izq).expand(B, -1)]
        for t in range(T):
            L.append(_normaliza(torch.einsum("ba,bac->bc", L[-1], Ax[:, t])))
        R = [_normaliza(self.der).expand(B, -1)]
        for t in reversed(range(T)):
            R.append(_normaliza(torch.einsum("bac,bc->ba", Ax[:, t], R[-1])))
        R = R[::-1]  # R[t] = entorno derecho que empieza en la posición t
        # La lectura de la posición t vive en el enlace (t, t+1): entorno izquierdo que YA incluye x_t
        # y entorno derecho a partir de t+1. Así cada bit de salida ve la secuencia entera.
        Lt = torch.stack(L[1:], dim=1)   # (B, T, χ): entorno izquierdo hasta t inclusive
        Rt = torch.stack(R[1:], dim=1)   # (B, T, χ): entorno derecho después de t
        return torch.einsum("nta,cab,ntb->ntc", Lt, self.M, Rt)

    def estado_final(self, x: torch.Tensor) -> torch.Tensor:
        Ax = self.por_posicion(x)
        v = _normaliza(self.izq).expand(Ax.shape[0], -1)
        for t in range(Ax.shape[1]):
            v = _normaliza(torch.einsum("ba,bac->bc", v, Ax[:, t]))
        return v


def born(s: torch.Tensor) -> torch.Tensor:
    """Amplitudes → log-probabilidades por la regla de Born (interferencia)."""
    if s.is_complex():
        p = s.real ** 2 + s.imag ** 2
        return torch.log(p + 1e-9) - torch.log(p.sum(-1, keepdim=True) + 1e-9)
    return F.log_softmax(s, dim=-1)


class ModeloMPS(nn.Module):
    """Una capa MPS con lectura de Born (o lineal si es real). Para 'parada' clasifica el estado final."""

    def __init__(self, chi: int, variante: str, secuencia: bool):
        super().__init__()
        self.secuencia = secuencia
        self.capa = CapaMPS(chi, 2, variante)
        if not secuencia:
            dt = torch.cfloat if variante != "real" else torch.float32
            self.lectura = nn.Parameter(torch.randn(2, chi, dtype=dt) / math.sqrt(chi))

    def forward(self, x):
        if self.secuencia:
            return born(self.capa(x))
        v = self.capa.estado_final(x)
        return born(torch.einsum("ba,ca->bc", v, self.lectura))


# ----------------------------------------------------------------------------- espigas (LIF)
class _Escalon(torch.autograd.Function):
    @staticmethod
    def forward(ctx, v):
        ctx.save_for_backward(v)
        return (v > 0).to(v.dtype)

    @staticmethod
    def backward(ctx, g):
        (v,) = ctx.saved_tensors
        sig = torch.sigmoid(4.0 * v)
        return g * 4.0 * sig * (1 - sig)  # gradiente sustituto


class CompuertaEspigas(nn.Module):
    """Integra-y-dispara con fuga a lo largo de la secuencia. Solo los canales cuya energía
    acumulada cruza el umbral 'disparan' y pasan a la capa siguiente; el resto queda en silencio.
    Devuelve la señal filtrada y la tasa de disparo (fracción de cómputo activo)."""

    def __init__(self, d: int, beta: float = 0.7):
        super().__init__()
        self.beta = beta
        self.umbral = nn.Parameter(torch.ones(d) * 0.5)

    def forward(self, f: torch.Tensor):
        B, T, d = f.shape
        e = f ** 2
        v = torch.zeros(B, d, dtype=f.dtype, device=f.device)
        salidas, tasa = [], 0.0
        for t in range(T):
            v = self.beta * v + e[:, t]
            s = _Escalon.apply(v - F.softplus(self.umbral))
            v = v * (1 - s)  # reinicio tras disparar
            salidas.append(s * f[:, t])
            tasa = tasa + s.mean()
        return torch.stack(salidas, dim=1), tasa / T


class ModeloMPSEspigas(nn.Module):
    """Dos capas MPS con una compuerta de espigas entre ambas. Con `compuerta=False` es el control:
    las mismas dos capas sin compuerta (la señal pasa entera)."""

    def __init__(self, chi: int, variante: str, d_oculto: int = 8, compuerta: bool = True):
        super().__init__()
        self.capa1 = CapaMPS(chi, d_oculto, variante)
        self.espigas = CompuertaEspigas(2 * d_oculto) if compuerta else None
        self.capa2 = CapaMPS(chi, 2, variante, entrada_continua=True, d_entrada=2 * d_oculto)
        self.tasa = torch.tensor(1.0)

    def forward(self, x):
        s = self.capa1(x)
        f = torch.cat([s.real, s.imag], dim=-1) if s.is_complex() else torch.cat([s, s * 0], -1)
        if self.espigas is not None:
            f, self.tasa = self.espigas(f)
        return born(self.capa2(f))


# ----------------------------------------------------------------------------- dos cabezas, crítica cruzada
class ModeloDual(nn.Module):
    """Dos cabezas MPS (A y B). Con crítica: cada cabeza ve la propuesta de la otra (sus amplitudes,
    módulo y fase) y emite una corrección compleja a la suya; la respuesta final es la suma
    (interferencia) de las dos amplitudes corregidas. Sin crítica: suma directa."""

    def __init__(self, chi: int, variante: str, critica: bool):
        super().__init__()
        self.critica = critica
        self.A = CapaMPS(chi, 2, variante)
        self.B = CapaMPS(chi, 2, variante)
        if critica:
            self.crit_A = nn.Sequential(nn.Linear(6, 16), nn.GELU(), nn.Linear(16, 4))
            self.crit_B = nn.Sequential(nn.Linear(6, 16), nn.GELU(), nn.Linear(16, 4))

    @staticmethod
    def _rasgos(s):
        return torch.cat([s.real, s.imag, s.real ** 2 + s.imag ** 2], dim=-1)

    @staticmethod
    def _corrige(s, c):
        return s + torch.complex(c[..., :2], c[..., 2:])

    def forward(self, x):
        sA, sB = self.A(x), self.B(x)
        self.parciales = (born(sA), born(sB))
        if self.critica:
            sA2 = self._corrige(sA, self.crit_A(self._rasgos(sB)))  # A critica a B y corrige su propuesta
            sB2 = self._corrige(sB, self.crit_B(self._rasgos(sA)))
            return born(sA2 + sB2)
        return born(sA + sB)


# ----------------------------------------------------------------------------- Transformer de referencia
def _rope(q, k, T, device):
    d = q.shape[-1]
    pos = torch.arange(T, device=device).float()
    inv = 1.0 / (10000 ** (torch.arange(0, d, 2, device=device).float() / d))
    ang = pos[:, None] * inv[None]
    cos, sin = ang.cos()[None, None], ang.sin()[None, None]

    def rot(x):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)
    return rot(q), rot(k)


class _Bloque(nn.Module):
    def __init__(self, d, h, rope):
        super().__init__()
        self.h, self.rope = h, rope
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.o = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, self.h, d // self.h).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = _rope(q, k, T, x.device)
        a = F.scaled_dot_product_attention(q, k, v)
        x = x + self.o(a.transpose(1, 2).reshape(B, T, d))
        return x + self.ffn(self.n2(x))


class Transformer(nn.Module):
    def __init__(self, d: int, capas: int, cabezas: int, posicion: str, secuencia: bool, max_pos: int = 128):
        super().__init__()
        self.posicion, self.secuencia, self.max_pos = posicion, secuencia, max_pos
        self.emb = nn.Embedding(VOCAB, d)
        if posicion == "abaco":
            self.pos = nn.Embedding(max_pos, d)
        self.bloques = nn.ModuleList([_Bloque(d, cabezas, rope=(posicion == "rope")) for _ in range(capas)])
        self.norma = nn.LayerNorm(d)
        self.salida = nn.Linear(d, 2)

    def forward(self, x):
        B, T = x.shape
        h = self.emb(x)
        if self.posicion == "abaco":
            desde = torch.randint(0, self.max_pos - T, (1,)).item() if self.training else 0
            h = h + self.pos(torch.arange(desde, desde + T, device=x.device))[None]
        for b in self.bloques:
            h = b(h)
        h = self.norma(h)
        if not self.secuencia:
            h = h.mean(1)
        return F.log_softmax(self.salida(h), dim=-1)


# ----------------------------------------------------------------------------- fábrica
def construir(nombre: str, chi: int, secuencia: bool) -> nn.Module:
    if nombre.startswith("mps-espigas"):
        return ModeloMPSEspigas(chi, "complejo")
    if nombre == "mps-2capas":  # control: dos capas sin compuerta
        return ModeloMPSEspigas(chi, "complejo", compuerta=False)
    if nombre == "mps-dual-critica":
        return ModeloDual(chi, "complejo", critica=True)
    if nombre == "mps-dual-suma":
        return ModeloDual(chi, "complejo", critica=False)
    if nombre.startswith("mps-"):
        return ModeloMPS(chi, nombre.split("-")[1], secuencia)
    if nombre.startswith("tf-"):  # χ=16 → d=64 y 2 capas; χ=32 → d=128 y 4 capas
        return Transformer(4 * chi, 2 if chi <= 16 else 4, 4, nombre.split("-")[1], secuencia)
    raise ValueError(nombre)


def n_parametros(m: nn.Module) -> int:
    return sum(p.numel() * (2 if p.is_complex() else 1) for p in m.parameters())


if __name__ == "__main__":
    torch.manual_seed(0)
    x = torch.randint(0, 3, (4, 10))
    for nombre in ["mps-real", "mps-complejo", "mps-unitario", "mps-fourier", "mps-espigas", "mps-dual-critica",
                   "mps-dual-suma", "tf-rope", "tf-abaco"]:
        m = construir(nombre, 16, True)
        y = m(x)
        assert y.shape == (4, 10, 2) and torch.isfinite(y).all(), nombre
        assert torch.allclose(y.exp().sum(-1), torch.ones(4, 10), atol=1e-4), nombre
        y.sum().backward()
        print(f"{nombre:18s} params={n_parametros(m):7d} salida={tuple(y.shape)}")
    U = cayley(torch.randn(8, 8, dtype=torch.cfloat))
    assert torch.allclose(U @ U.conj().T, torch.eye(8, dtype=torch.cfloat), atol=1e-5)
    m = construir("mps-complejo", 16, False)
    print("parada:", m(x).shape)
    print("ok")


In [ ]:
%%writefile experimentos.py
"""Entrenamiento y evaluación del banco de superposición.

    python investigacion/superposicion/experimentos.py --rapido          # una corrida de prueba
    python investigacion/superposicion/experimentos.py --plan base       # el plan completo, en paralelo (CPU)

Protocolo (el mismo que en navros/): se entrena con longitudes mezcladas 3..L_train, se elige nada
con la prueba fuera de distribución; las longitudes largas se miden UNA vez al final.
Cada corrida escribe un JSON en resultados/<plan>/ con la curva y las exactitudes por longitud.
"""
from __future__ import annotations

import argparse
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np

AQUI = Path(__file__).resolve().parent
sys.path.insert(0, str(AQUI))

L_TRAIN = 12
L_TEST = [12, 16, 24, 32]


def correr(modelo: str, tarea: str, chi: int = 16, semilla: int = 0, pasos: int = 3000, lote: int = 128,
           lr: float | None = None, L_train: int = L_TRAIN, L_test=tuple(L_TEST), n_eval: int = 512,
           hilos: int = 1, log=print) -> dict:
    import torch
    import collatz
    import modelos
    torch.set_num_threads(hilos)
    torch.manual_seed(semilla)
    disp = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rng = np.random.default_rng(semilla)
    secuencia = tarea.startswith("paso-")
    m = modelos.construir(modelo, chi, secuencia).to(disp)
    if lr is None:
        lr = 1e-3 if modelo.startswith("tf-") else 1e-2
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01, betas=(0.9, 0.98))
    calendario = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / 100) * 0.5 * (1 + math.cos(math.pi * min(s, pasos) / pasos)))
    t0, historia = time.time(), []
    m.train()
    for paso in range(pasos):
        L = int(rng.integers(3, L_train + 1))
        x, y = collatz.lote(tarea, rng, L, lote)
        x, y = torch.from_numpy(x).to(disp), torch.from_numpy(y).to(disp)
        logp = m(x)
        perdida = -logp.gather(-1, y.unsqueeze(-1)).mean()
        if hasattr(m, "parciales"):  # cabezas duales: cada cabeza debe ser competente por sí sola
            for lp in m.parciales:
                perdida = perdida + 0.5 * (-lp.gather(-1, y.unsqueeze(-1)).mean())
        opt.zero_grad(set_to_none=True)
        perdida.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        calendario.step()
        if paso % 100 == 0 or paso == pasos - 1:
            with torch.no_grad():
                pred = logp.argmax(-1)
                ex = (pred == y).all(-1).float().mean().item() if secuencia else (pred == y).float().mean().item()
            h = dict(paso=paso, L=L, perdida=round(perdida.item(), 4), exacta=round(ex, 4), t=round(time.time() - t0, 1))
            if hasattr(m, "tasa"):
                h["tasa_disparo"] = round(float(m.tasa.detach()), 4)
            historia.append(h)
            log(f"  {modelo} {tarea} χ={chi} s={semilla} " + " ".join(f"{k}={v}" for k, v in h.items()))
    m.eval()
    evals = {}
    rng_eval = np.random.default_rng(10_000 + semilla)
    with torch.no_grad():
        for L in L_test:
            x, y = collatz.lote(tarea, rng_eval, L, n_eval)
            x, y = torch.from_numpy(x).to(disp), torch.from_numpy(y).to(disp)
            pred = m(x).argmax(-1)
            if secuencia:
                evals[str(L)] = dict(exacta=(pred == y).all(-1).float().mean().item(),
                                     bits=(pred == y).float().mean().item())
            else:
                evals[str(L)] = dict(exacta=(pred == y).float().mean().item())
    return dict(modelo=modelo, tarea=tarea, chi=chi, semilla=semilla, pasos=pasos, lote=lote, lr=lr,
                L_train=L_train, parametros=modelos.n_parametros(m), segundos=round(time.time() - t0, 1),
                historia=historia, eval=evals)


# ----------------------------------------------------------------------------- planes
def plan(nombre: str, semillas=(0, 1, 2)) -> list[dict]:
    cfgs = []
    if nombre in ("base", "todo"):
        for tarea in ["paso-1", "paso-2", "paso-3"]:
            for modelo in ["mps-real", "mps-complejo", "mps-unitario", "mps-fourier", "tf-rope", "tf-abaco"]:
                for s in semillas:
                    cfgs.append(dict(modelo=modelo, tarea=tarea, chi=16, semilla=s))
        for modelo in ["mps-complejo", "tf-rope"]:
            for s in semillas:
                cfgs.append(dict(modelo=modelo, tarea="parada", chi=16, semilla=s))
    if nombre in ("extras", "todo"):
        for modelo in ["mps-espigas", "mps-dual-critica", "mps-dual-suma"]:
            for s in semillas:
                cfgs.append(dict(modelo=modelo, tarea="paso-2", chi=16, semilla=s))
        for modelo in ["mps-complejo", "mps-fourier"]:
            for s in semillas:
                cfgs.append(dict(modelo=modelo, tarea="paso-2", chi=32, semilla=s))
    if nombre in ("extras2", "todo"):  # crítica cruzada y espigas donde χ=16 NO satura: paso-3
        for modelo in ["mps-dual-critica", "mps-dual-suma", "mps-espigas"]:
            for s in semillas:
                cfgs.append(dict(modelo=modelo, tarea="paso-3", chi=16, semilla=s))
    if nombre == "profundidad":  # ¿el 100 % en paso-3 era de la profundidad o del silencio?
        for modelo in ["mps-complejo", "mps-2capas", "mps-espigas"]:
            for s in semillas:
                cfgs.append(dict(modelo=modelo, tarea="paso-3", chi=16, semilla=s))
    if nombre == "escala":  # escala ordinal: mayor k resuelto exacto y general por cada χ
        for chi in (8, 16, 32, 64):
            for k in range(1, 7):
                for s in semillas:
                    cfgs.append(dict(modelo="mps-complejo", tarea=f"paso-{k}", chi=chi, semilla=s))
        for chi in (16, 32):  # Transformer d=64 y d=128 como referencia
            for k in range(1, 7):
                for s in semillas:
                    cfgs.append(dict(modelo="tf-rope", tarea=f"paso-{k}", chi=chi, semilla=s))
    if nombre == "escala-local":  # versión reducida para una CPU: 1 semilla, χ ≤ 32, k ≤ 5
        for chi in (8, 16, 32):
            for k in range(1, 6):
                cfgs.append(dict(modelo="mps-complejo", tarea=f"paso-{k}", chi=chi, semilla=0))
    if nombre == "grande":  # para GPU/Modal: más ancho, más largo, más pasos; L_train=16, prueba hasta 48
        for tarea in ["paso-1", "paso-2", "paso-3"]:
            for modelo in ["mps-complejo", "mps-unitario", "mps-fourier", "mps-real", "tf-rope", "tf-abaco",
                           "mps-dual-critica", "mps-dual-suma", "mps-espigas"]:
                for s in semillas:
                    cfgs.append(dict(modelo=modelo, tarea=tarea, chi=32, semilla=s, pasos=6000,
                                     L_train=16, L_test=(16, 24, 32, 48)))
        for modelo in ["mps-complejo", "mps-unitario", "tf-rope"]:
            for s in semillas:
                cfgs.append(dict(modelo=modelo, tarea="parada", chi=32, semilla=s, pasos=6000,
                                 L_train=16, L_test=(16, 24, 32, 48)))
    if not cfgs:
        raise ValueError(nombre)
    return cfgs


def etiqueta(c: dict) -> str:
    return f"{c['modelo']}_{c['tarea']}_chi{c['chi']}_s{c['semilla']}"


def _trabajo(args):
    c, carpeta = args
    ruta = Path(carpeta) / (etiqueta(c) + ".json")
    if ruta.exists():
        return json.loads(ruta.read_text())
    res = correr(**c, log=lambda s: None)  # noqa
    ruta.write_text(json.dumps(res, indent=1))
    print(f"listo {etiqueta(c)} {res['segundos']}s eval={ {k: round(v['exacta'], 3) for k, v in res['eval'].items()} }", flush=True)
    return res


def ejecutar_plan(nombre: str, carpeta: Path, procesos: int):
    from multiprocessing import Pool
    carpeta.mkdir(parents=True, exist_ok=True)
    cfgs = plan(nombre)
    print(f"plan {nombre}: {len(cfgs)} corridas, {procesos} procesos", flush=True)
    with Pool(procesos) as p:
        return list(p.imap_unordered(_trabajo, [(c, str(carpeta)) for c in cfgs]))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--rapido", action="store_true")
    ap.add_argument("--plan", default=None)
    ap.add_argument("--procesos", type=int, default=max(1, (os.cpu_count() or 2)))
    ap.add_argument("--modelo", default="mps-complejo")
    ap.add_argument("--tarea", default="paso-1")
    ap.add_argument("--pasos", type=int, default=300)
    a = ap.parse_args()
    if a.rapido:
        r = correr(a.modelo, a.tarea, pasos=a.pasos, hilos=os.cpu_count() or 1)
        print(json.dumps(r["eval"], indent=1), r["segundos"], "s")
    elif a.plan:
        ejecutar_plan(a.plan, AQUI / "resultados" / a.plan, a.procesos)


In [ ]:
%%writefile resumen.py
"""Agrega los JSON de resultados/<plan>/ en tablas Markdown: media ± desv. típica sobre semillas.

    python resumen.py base extras
"""
from __future__ import annotations

import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np

AQUI = Path(__file__).resolve().parent


def cargar(planes):
    filas = []
    for p in planes:
        for f in sorted((AQUI / "resultados" / p).glob("*.json")):
            filas.append(json.loads(f.read_text()))
    return filas


def tabla(filas, tarea, metrica="exacta"):
    grupos = defaultdict(list)
    for r in filas:
        if r["tarea"] == tarea:
            grupos[(r["modelo"], r["chi"])].append(r)
    if not grupos:
        return ""
    Ls = sorted({L for r in filas if r["tarea"] == tarea for L in r["eval"]}, key=int)
    out = [f"### {tarea} — exactitud de secuencia completa (%), media ± desv. típica, {max(len(v) for v in grupos.values())} semillas\n",
           "| modelo | χ / d | parámetros | " + " | ".join(f"L={L}" + (" (entreno)" if L == str(filas[0]["L_train"]) else "") for L in Ls) + " | s/corrida |",
           "|---|---|---|" + "---|" * len(Ls) + "---|"]
    for (m, chi), rs in sorted(grupos.items()):
        celdas = []
        for L in Ls:
            v = np.array([r["eval"][L][metrica] for r in rs]) * 100
            celdas.append(f"{v.mean():.1f} ± {v.std():.1f}")
        extra = ""
        if any("tasa_disparo" in h for h in rs[0]["historia"]):
            t = np.mean([r["historia"][-1].get("tasa_disparo", 0) for r in rs]) * 100
            extra = f" (disparo {t:.0f} %)"
        out.append(f"| {m}{extra} | {chi} | {rs[0]['parametros']:,} | " + " | ".join(celdas) + f" | {np.mean([r['segundos'] for r in rs]):.0f} |")
    return "\n".join(out) + "\n"


if __name__ == "__main__":
    planes = sys.argv[1:] or ["base"]
    filas = cargar(planes)
    print(f"{len(filas)} corridas de {planes}\n")
    for tarea in ["paso-1", "paso-2", "paso-3", "parada"]:
        t = tabla(filas, tarea)
        if t:
            print(t)
    t = tabla(filas, "paso-2", "bits")
    if t:
        print(t.replace("exactitud de secuencia completa", "exactitud por bit"))


In [ ]:
import subprocess, sys, os, time
os.makedirs("resultados", exist_ok=True)
t = time.time()
procs = []
for gpu, plan in [(0, "profundidad"), (1, "escala")]:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"resultados/{plan}.log", "w")
    procs.append((plan, subprocess.Popen([sys.executable, "experimentos.py", "--plan", plan, "--procesos", "2"],
                                         env=env, stdout=log, stderr=subprocess.STDOUT)))
    print(f"GPU {gpu}: plan {plan} lanzado", flush=True)
for plan, p in procs:
    p.wait()
    print(f"{plan} terminado con código {p.returncode} a los {(time.time()-t)/60:.1f} min", flush=True)


In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "resumen.py", "profundidad"], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, "resumen.py", "escala"], capture_output=True, text=True).stdout)
